# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/emManab/flyrankintern/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.


## 1. Two signal checks

My baseline uses two pre-decision signals: **staleness** (`days_since_last_update`) and **visibility/volume** (`impressions_90d`). Staleness is directly linked to FlyRank's refresh/staleness flag logic; volume is linked to the quick-win/visible-opportunity logic. The outcome label is `trend_direction == "down"`, but `trend_direction` and `trend_pct` are never used as features.

In [ ]:
from pathlib import Path
import pandas as pd, numpy as np
DATA=Path("data/raw/content_refresh_anonymized.csv")
if not DATA.exists(): DATA=next(Path.cwd().rglob("content_refresh_anonymized.csv"))
df=pd.read_csv(DATA)
df["is_declining_label"]=df["trend_direction"].astype(str).str.lower().eq("down").astype(int)
print("rows:",len(df),"columns:",len(df.columns),"clients:",df["client_id"].nunique())

# Signal 1: staleness, linked to FlyRank's stale/refresh flag.
df["stale_bucket"]=pd.cut(df["days_since_last_update"],[-np.inf,89,179,269,np.inf],labels=["<90d","90-179d","180-269d","270d+"])
stale_tab=df.groupby("stale_bucket",observed=False).agg(n=("content_id","size"),declining_rate=("is_declining_label","mean"),median_impressions=("impressions_90d","median")).reset_index()
print("\nSIGNAL 1 — days_since_last_update")
print(stale_tab.to_string(index=False,formatters={"declining_rate":"{:,.3f}".format,"median_impressions":"{:,.0f}".format}))
stale_ref=stale_tab.loc[stale_tab["stale_bucket"].isin(["180-269d","270d+"]),"declining_rate"].mean()
fresh_ref=stale_tab.loc[stale_tab["stale_bucket"].isin(["<90d","90-179d"]),"declining_rate"].mean()
stale_verdict="CONFIRMED" if stale_ref > fresh_ref else "OPPOSITE" if stale_ref < fresh_ref else "MIXED"
print(f"VERDICT: {stale_verdict} — mean declining rate in 180d+ buckets={stale_ref:.3f} vs <180d={fresh_ref:.3f}.")

# Signal 2: search volume / visibility, linked to the visible-opportunity / quick-win logic.
df["volume_bucket"]=pd.cut(df["impressions_90d"],[0,99,299,999,2999,9999,np.inf],right=True,labels=["1-99","100-299","300-999","1k-2.9k","3k-9.9k","10k+"])
vol_tab=df.groupby("volume_bucket",observed=False).agg(n=("content_id","size"),declining_rate=("is_declining_label","mean"),median_days_stale=("days_since_last_update","median")).reset_index()
print("\nSIGNAL 2 — impressions_90d")
print(vol_tab.to_string(index=False,formatters={"declining_rate":"{:,.3f}".format,"median_days_stale":"{:,.0f}".format}))
lo=vol_tab.loc[vol_tab["volume_bucket"].isin(["1-99","100-299"]),"declining_rate"].mean()
hi=vol_tab.loc[vol_tab["volume_bucket"].isin(["3k-9.9k","10k+"]),"declining_rate"].mean()
vol_verdict="CONFIRMED" if hi >= lo else "OPPOSITE" if hi < lo and abs(hi-lo) >= 0.05 else "MIXED"
print(f"VERDICT: {vol_verdict} — mean declining rate in 3k+ buckets={hi:.3f} vs under 300={lo:.3f}.")


### Rule reasoning

The baseline should remain simple and human-readable. I use **stale + visible** as the rule: a page is a priority when it is at least 180 days since its last update and has at least 500 search impressions in the 90-day window. The score is exposure (`impressions_90d`) for qualifying pages and 0 otherwise. Every non-zero item gets one reason code, `stale_visible_page`.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write `work/outputs/baseline_action_score.csv`.*

In [ ]:
# One transparent rule: stale + visible, ranked by observed exposure.
stale=(df["days_since_last_update"]>=180)
visible=(df["impressions_90d"]>=500)
df["score"]=np.where(stale & visible, df["impressions_90d"], 0).astype(float)
df["reason_code"]=np.where(df["score"]>0,"stale_visible_page","not_selected")
df["action_label"]=np.where(df["score"]>0,"refresh","monitor")
df["baseline_rank"]=df["score"].rank(method="first",ascending=False).astype(int)
queue_cols=["content_id","client_id","baseline_rank","score","reason_code","action_label","impressions_90d","days_since_last_update","avg_position","ctr","sessions_90d","content_age_days","word_count","trend_direction","is_declining_label"]
queue=df[queue_cols].sort_values("baseline_rank").reset_index(drop=True)
OUT=Path("work/outputs/baseline_action_score.csv")
OUT.parent.mkdir(parents=True,exist_ok=True)
queue.to_csv(OUT,index=False)

def precision_at_k(scores,labels,k):
    order=np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

base_rate=df["is_declining_label"].mean()
print("queue rows:",len(queue))
print("flagged rows:",int((df["score"]>0).sum()))
print(f"base declining rate: {base_rate:.3f}")
for k in (20,50):
    print(f"Precision@{k}: {precision_at_k(df['score'],df['is_declining_label'],k):.3f}")
print("wrote:",OUT)


## 3. Top-10 review

Each row below records the action, why it was selected, and what would make the recommendation wrong. The last field is deliberately skeptical: the baseline is decision support, not an automatic publishing decision.

In [ ]:
top10=queue.head(10).copy()
top10["why"]="stale >=180d and visible >=500 impressions; higher impressions rank higher"
top10["what_would_make_it_wrong"]=(
    "The page was refreshed after this snapshot, the impressions are not current, "
    "or the traffic is not commercially/actionably valuable."
)
print(top10[["baseline_rank","content_id","action_label","reason_code","score","why","what_would_make_it_wrong"]].to_string(index=False))


## 4. Weak picks + leakage check

A weak pick is a page that scores highly because it is stale and visible but may not be a useful refresh target. For example, a page can have many impressions but weak business value, poor intent fit, or already have a recent manual refresh outside the measured field.

Leakage check: the score uses only `days_since_last_update` and `impressions_90d`. It does **not** use `trend_direction`, `trend_pct`, IDs, product flags, or any future-window outcome. `is_declining_label` is retained only for evaluation.

In [ ]:
weak=queue.head(3).copy()
print("Weak-pick review: high-volume pages can still be poor recommendations when business value or recent manual context is missing.")
print(weak[["baseline_rank","content_id","score","action_label","reason_code"]].to_string(index=False))
print("LEAKAGE CHECK: features = ['days_since_last_update','impressions_90d']; outcome/label fields are excluded from scoring.")


## Self-check

- [x] Two signal verdicts are visible with bucket tables and `n`.
- [x] At least one signal is linked to a real FlyRank flag.
- [x] One transparent rule with a score, one reason code, and an action label.
- [x] Ranked queue is written by the notebook to `work/outputs/baseline_action_score.csv`.
- [x] Top-10 review includes action, why, and what would make it wrong.
- [x] No future-window or label-derived inputs are used as score features.
- [ ] Notebook must still be executed in Colab/local and committed with outputs.
